# LLM Inference Concepts

**Module:** 05 — LLM Fundamentals

Autoregressive decoding, prefill vs decode, and batching—including continuous batching.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain autoregressive generation loops
- Contrast prefill and decode compute patterns
- Describe batching and continuous batching benefits
- Relate concepts to latency/cost tradeoffs


## Autoregressive Decoding

**Definition.** Generate tokens one-by-one: each new token conditions on all previous tokens.

**Why it matters.** This is why generation latency grows with output length and why streaming works.

**How it works.** Prefill prompt → repeatedly sample next token until stop/max length.

**Intuition.** Writing a story a word at a time, rereading everything so far (via cache).

**Common pitfalls.**
- Requesting huge max tokens 'just in case'
- Ignoring stop sequences

**When to use.** All standard chat/completions APIs.

```mermaid
flowchart LR
  P[Prompt] --> Prefill
  Prefill --> D1[Decode tok1]
  D1 --> D2[Decode tok2]
  D2 --> Dn[...]
```


In [ ]:
# Demo 1 — AR loop pseudo
def generate(prompt_tokens, next_token_fn, max_new=10):
    toks = list(prompt_tokens)
    for _ in range(max_new):
        nxt = next_token_fn(toks)
        toks.append(nxt)
        if nxt == "</s>": break
    return toks
print(generate(["<bos>", "Hi"], lambda t: "</s>" if len(t)>3 else "there"))


In [ ]:
# Demo 2 — latency model
ttft, tps, out = 0.4, 60, 120  # sec, tokens/s, tokens
print("approx total sec", ttft + out/tps)


In [ ]:
# Demo 3 — streaming chunks
chunks = ["Hel", "lo", "!", " </s>"]
for c in chunks:
    print("event:", repr(c))


### Try it yourself — Autoregressive Decoding

1. Estimate time-to-last-token for 800 output tokens at 40 tok/s with 0.5s TTFT.


## Prefill vs Decode

**Definition.** **Prefill** processes the full prompt in parallel to build KV cache; **decode** appends one token at a time.

**Why it matters.** Optimizations differ: prefill is compute-heavy/bound; decode often memory-bandwidth bound.

**How it works.** Run large matmuls over all prompt positions once; then reuse KV for cheap per-token steps.

**Intuition.** Read the whole brief, then write the reply word by word.

**Common pitfalls.**
- Blaming decode slowness when prompt prefill dominates
- Not caching system prompts when possible

**When to use.** Performance engineering and capacity planning.


In [ ]:
# Demo 1 — FLOPs-ish comparison
prompt_T, new_T, D = 2000, 100, 4096
prefill = prompt_T  # units of "token process"
decode = new_T
print({"prefill_units": prefill, "decode_units": decode})


In [ ]:
# Demo 2 — KV cache growth
layers, heads, hd, dtype_bytes = 32, 32, 128, 2
def kv_bytes(tokens):
    # K+V per layer
    return tokens * layers * 2 * heads * hd * dtype_bytes
print(kv_bytes(4096)/1e6, "MB")


In [ ]:
# Demo 3 — prefix caching idea
print("static system prompt → cache KV once → reuse across requests")


### Try it yourself — Prefill vs Decode

1. Which phase dominates for a 50-token answer to a 20k-token RAG prompt?


## Batching & Continuous Batching

**Definition.** **Batching** groups requests to utilize GPUs; **continuous batching** lets new requests join as others finish tokens.

**Why it matters.** Critical for serving throughput and cost per token.

**How it works.** Iteration-level schedulers (vLLM-style) keep GPUs full despite variable lengths.

**Intuition.** A restaurant seating parties as tables free, not waiting for everyone to finish.

**Common pitfalls.**
- Static batching with huge padding waste
- Starvation without fairness policies

**When to use.** Any multi-tenant LLM service.


In [ ]:
# Demo 1 — static batch padding waste
lengths = [5, 40, 12]
padded = max(lengths) * len(lengths)
real = sum(lengths)
print({"padded": padded, "real": real, "waste_frac": 1 - real/padded})


In [ ]:
# Demo 2 — continuous batching timeline
events = ["reqA start", "reqB join", "reqA done", "reqC join"]
for e in events:
    print("→", e)


In [ ]:
# Demo 3 — throughput vs latency tradeoff
print("| batch | throughput | latency |")
print("| small | lower | better p50 |")
print("| large | higher | worse p99 |")


### Try it yourself — Batching & Continuous Batching

1. Explain continuous batching to an SRE in three sentences.


## Glossary

- **TTFT**: Time to first token
- **KV cache**: Cached keys/values for prior tokens


### Workshop drill — LLM Inference Concepts (1)

Restate each section heading as a single exam-ready sentence.


In [ ]:
# Workshop drill 1 — LLM Inference Concepts
headings = ['Autoregressive Decoding', 'Prefill vs Decode', 'Batching & Continuous Batching']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — LLM Inference Concepts (2)

Change one hyperparameter/assumption in a demo and predict the effect before running.


In [ ]:
# Workshop drill 2 — LLM Inference Concepts
print('prediction: ...')
print('observation: ...')
print('delta: ...')


### Workshop drill — LLM Inference Concepts (3)

List production risks (cost, latency, safety, quality) for this topic.


In [ ]:
# Workshop drill 3 — LLM Inference Concepts
for r in ['cost','latency','safety','quality']:
    print(f'{r}:')


### Workshop drill — LLM Inference Concepts (4)

Write a tiny unit-testable helper related to the lesson and assert two cases.


In [ ]:
# Workshop drill 4 — LLM Inference Concepts
def ok(x):
    return x is not None
assert ok(1) and not ok(None)
print('ok')


### Workshop drill — LLM Inference Concepts (5)

Sketch an API request/response JSON for a realistic call tied to this topic.


In [ ]:
# Workshop drill 5 — LLM Inference Concepts
import json
print(json.dumps({'model':'...','input':'...','output':'...'}, indent=2))


### Workshop drill — LLM Inference Concepts (6)

Compare two design alternatives in a markdown table (fill TODOs).


In [ ]:
# Workshop drill 6 — LLM Inference Concepts
print('| option | pros | cons |')
print('|--------|------|------|')
print('| A | TODO | TODO |')
print('| B | TODO | TODO |')


## Summary & Key Takeaways

- Autoregressive decode emits tokens sequentially
- Prefill ≠ decode for bottlenecks and caching
- Continuous batching is key to efficient serving

### Practice

Measure TTFT vs tokens/sec on a provider playground if you have access.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
